# PydanticAT together with Pydantic Models

In [ ]:
from model import nvidia_model
from pydantic_ai import Agent

employee_simulator = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige. Simulera i detta formatet likt exemplet"
    """
        name: "Bertil",
        age: 40,
        salary: 35000,
        role: "IT-supporter",
        city: "Göteborg"
    """
    )

result = await employee_simulator.run("Ge mig en data engineer som jobbar i stockholm")
print(result.output)

        name: "Erik",
        age: 34,
        salary: 54000,
        role: "Data engineer",
        ort: "Stockholm"


## Forece a structured output using pydantic

In [3]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str = Field(description="för och efternamn, inte jättelångt, svenska eller internationella namn")
    age: int = Field(description="ålder från 18 till 67 år", ge=18, le=67)
    salary: int = Field(description="lön mellan 30k-60k, ju mer senior position, desto högre lön", 
                        ge=30_000, 
                        le=60_000
                        )
    role:str = Field(description="yrke/roll")
    description: str = Field(description="kort beskrivning, 2 meningar på rollen")


employee1 = EmployeeModel(name="Bertil", age=40, salary=30_000, role="IT-supporter", description="Hjälper folk med IT")

employee1

EmployeeModel(name='Bertil', age=40, salary=30000, role='IT-supporter', description='Hjälper folk med IT')

In [9]:
employee_simulator = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions="Du är expert på att simulera anställda i Sverige."
)

result = await employee_simulator.run("Ge mig en data engineer i stockholm", output_type=EmployeeModel)

employee2 = result.output
employee2

EmployeeModel(name='Sofia Lindberg', age=32, salary=48000, role='Data Engineer', description='Arbetar som Data Engineer i Stockholm med att designa och underhålla dataplattformer i molnet. Har erfarenhet av SQL, Python och moderna ETL-verktyg för att stödja datadrivna beslut.')

In [10]:
type(employee2)

__main__.EmployeeModel

In [14]:
employee2.name + " är cool"

'Sofia Lindberg är cool'

In [15]:
employee2.salary

48000

In [17]:
employee2.__dict__.keys()

dict_keys(['name', 'age', 'salary', 'role', 'description'])

In [18]:
employee2.__dict__.values()

dict_values(['Sofia Lindberg', 32, 48000, 'Data Engineer', 'Arbetar som Data Engineer i Stockholm med att designa och underhålla dataplattformer i molnet. Har erfarenhet av SQL, Python och moderna ETL-verktyg för att stödja datadrivna beslut.'])

In [19]:
for key, value in employee2.__dict__.items():
    print(f"{key}: {value}")

name: Sofia Lindberg
age: 32
salary: 48000
role: Data Engineer
description: Arbetar som Data Engineer i Stockholm med att designa och underhålla dataplattformer i molnet. Har erfarenhet av SQL, Python och moderna ETL-verktyg för att stödja datadrivna beslut.


## Simulate many employees

In [ ]:
result = await employee_simulator.run("Ge mig totalt 10 anställda inom IT/HR/marketing",
                                      output_type=list[EmployeeModel])

result

In [ ]:
employees = result.output

employees

In [48]:
salary = 0

for employee in employees:    
    salary += employee.salary

salary/len(employees)

45300.0

## Convert to datafarame

In [49]:
employees[0].model_dump()

{'name': 'Emma Svensson',
 'age': 26,
 'salary': 38000,
 'role': 'Utvecklare',
 'description': 'Emma arbetar som juniorutvecklare och fokuserar främst på frontend-utveckling med React. Hon är nyutexaminerad och lär sig snabbt i teamet.'}

In [52]:
import pandas as pd
df = pd.DataFrame([employee.model_dump() for employee in employees])
df.head()

,name,age,salary,role,description
0,Emma Svensson,26,38000,Utvecklare,Emma arbetar som juniorutvecklare och fokusera...
1,Johan Karlsson,42,58000,Seniorutvecklare,Johan är seniorutvecklare med specialisering i...
2,Sara Andersson,35,45000,Systemadministratör,Sara ansvarar för företagets IT-infrastruktur ...
3,Lisa Nilsson,29,34000,HR-assistent,Lisa stödjer HR-avdelningen med administrativa...
4,Mikael Berg,48,56000,HR-chef,Mikael leder HR-avdelningen och arbetar strate...


In [53]:
df.to_csv("employees.csv", index=None)